In [4]:
import torch
import torch.nn as nn
import random

In [10]:
class LuongAttention(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, decoder_hidden, encoder_outputs):
    decoder_hidden = decoder_hidden.permute(1, 0, 2)

    scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1,2))

    alfa = torch.softmax(scores, dim=-1)

    context = torch.bmm(alfa, encoder_outputs)

    return context

In [8]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_size):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)

  def forward(self, x):
    out, hn = self.gru(self.embed(x))

    return out, hn

# Decoder

class Decoder(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_size, contextGenerator):
    super().__init__()
    self.vocab_size = vocab_size
    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)
    self.lin = nn.Linear(hidden_size, vocab_size)
    self.context_generator = contextGenerator

  def forward_step(self, input_token, hidden, encoder_out):
    embeded = self.embed(input_token).unsqueeze(1)

    out, hidden2 = self.gru(embeded, hidden)

    context = self.context_generator(hidden2, encoder_out)

    out = out + context

    logits = self.lin(out.squeeze(1))

    return logits, hidden2


In [9]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder, teacher_forcing=0.5):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder
    self.teacher_forcing = teacher_forcing

  def forward(self, x, target):
    outs, hidden = self.encoder(x)
    #print(x.shape)
    #print(target.shape)

    batch_size, target_len = target.shape

    decoder_input = target[:, 0]

    outputs = torch.zeros(batch_size, target_len, self.decoder.vocab_size)

    for t in range(1, target_len):
      logits, hidden = self.decoder.forward_step(decoder_input, hidden, outs)

      predicted = torch.argmax(logits, dim=1)

      outputs[:, t] = logits

      if random.uniform(0.1, 1.0) < self.teacher_forcing:
        decoder_input = target[:, t]

      else:
        decoder_input = predicted

    return outputs


In [ ]:
contextGenerator = LuongAttention()
encoder = Encoder(len(upper_to_idx), 16, 64)
decoder = Decoder(len(lower_to_idx), 16, 64, contextGenerator)